# Лабораторная работа 13. Основы Torch

PyTorch — один из самых известных фреймворков для машинного обучения и разработки нейросетей на языке Python. Его создала команда Meta AI (Facebook).

Почему его любят: Он позволяет строить нейросети «на лету» (динамические графы), что делает отладку и эксперименты гораздо проще, чем в других инструментах.

Где применяется: В компьютерном зрении, распознавании речи, машинном переводе и генерации текстов (например, на нем работают алгоритмы Instagram и Tesla).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# Создание тензоров
# Из списка
t1 = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)
print(t1)

tensor([[1., 2.],
        [3., 4.]])


In [ ]:
# Случайные числа (нормальное распределение)
t2 = torch.randn(2, 3)
print(t2)

tensor([[ 0.6119, -0.0895, -0.6442],
        [-0.1640,  1.1245, -0.6327]])


In [ ]:
# Тензор из нулей
zeros = torch.zeros(2, 2)
# Тензор из единиц
ones = torch.ones(3, 1)
# Единичная матрица
identity = torch.eye(3)
print(zeros)
print(ones)
print(identity)

tensor([[0., 0.],
        [0., 0.]])
tensor([[1.],
        [1.],
        [1.]])
tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])


In [ ]:
t1 = t1.to(device='cuda')
print(f"Тензор t1:\n{t1}\nТип данных: {t1.dtype}, Устройство: {t1.device}")

Тензор t1:
tensor([[1., 2.],
        [3., 4.]], device='cuda:0')
Тип данных: torch.float32, Устройство: cuda:0


In [ ]:
t1 = t1.to(device='cpu')
print(f"Тензор t1:\n{t1}\nТип данных: {t1.dtype}, Устройство: {t1.device}")

Тензор t1:
tensor([[1., 2.],
        [3., 4.]])
Тип данных: torch.float32, Устройство: cpu


In [ ]:
# Изменение формы (Reshape & View)
x = torch.arange(12) # [0, 1, 2 ... 11]

print(x)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])


In [ ]:
# view() и reshape() делают похожее,
# но view работает только с непрерывными данными в памяти
x_3x4 = x.view(3, 4)
x_2x2x3 = x.reshape(2, 2, 3)

print(x_2x2x3)

# Использование -1: Torch сам вычислит размерность
x_auto = x.view(2, -1) # Станет (2, 6)

tensor([[[ 0,  1,  2],
         [ 3,  4,  5]],

        [[ 6,  7,  8],
         [ 9, 10, 11]]])


In [ ]:
x = torch.arange(12) # [0, 1, 2 ... 11]
x_unsqueezed = x.unsqueeze(0).unsqueeze(0).unsqueeze(0)
print(x_unsqueezed)

tensor([[[[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11]]]])


In [ ]:
x = x_unsqueezed.squeeze()
print(x)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])


In [ ]:
# Squeeze/Unsqueeze (добавление/удаление единичных размерностей)
y = torch.randn(3, 3)
print(y)
y_unsqueezed = y.unsqueeze(0) # Станет (1, 3, 3) - часто нужно для batch_size
print(y_unsqueezed)
y_final = y_unsqueezed.squeeze() # Уберет все размерности равные 1

print(f"Было: {x.shape}, Стало (view 3x4): {x_3x4.shape}")

tensor([[-0.9697,  1.3015, -0.0757],
        [-1.2959, -0.0176, -0.8719],
        [-1.7848,  0.2773,  1.2497]])
tensor([[[-0.9697,  1.3015, -0.0757],
         [-1.2959, -0.0176, -0.8719],
         [-1.7848,  0.2773,  1.2497]]])
Было: torch.Size([12]), Стало (view 3x4): torch.Size([3, 4])


In [ ]:
# Математические операции
a = torch.tensor([10, 20, 30])
b = torch.tensor([1, 2, 3])

# Поэлементные операции
print(f"Сложение: {a + b}")
print(f"Умножение на скаляр: {a * 2}")

Сложение: tensor([11, 22, 33])
Умножение на скаляр: tensor([20, 40, 60])


In [ ]:
print(a)

tensor([10, 20, 30])


In [ ]:
# Матричное умножение (самое важное в Deep Learning)
mat1 = torch.randn(2, 3)
mat2 = torch.randn(3, 4)

# Три способа сделать одно и то же:
res1 = torch.mm(mat2.t(), mat1.T).T
res2 = mat1 @ mat2
res3 = torch.matmul(mat1, mat2) # matmul поддерживает broadcasting (разные размерности)
print(res1)
print(res2)
print(res3)

tensor([[-0.1323,  0.3396,  0.1080, -0.0782],
        [-2.6732, -1.5005, -0.5573,  2.4181]])
tensor([[-0.1323,  0.3396,  0.1080, -0.0782],
        [-2.6732, -1.5005, -0.5573,  2.4181]])
tensor([[-0.1323,  0.3396,  0.1080, -0.0782],
        [-2.6732, -1.5005, -0.5573,  2.4181]])


In [ ]:
# Агрегация и оси
m = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)

print(f"Общая сумма: {m.sum()}")
print(f"Сумма по столбцам (dim=0): {m.sum(dim=0)}")
print(f"Среднее по строкам (dim=1): {m.mean(dim=1)}")
print(f"Индекс максимального элемента: {m.argmax()}")

Общая сумма: 21.0
Сумма по столбцам (dim=0): tensor([5., 7., 9.])
Среднее по строкам (dim=1): tensor([2., 5.])
Индекс максимального элемента: 5


In [ ]:
# Слайсинг и индексация
z = torch.randn(4, 4)
print(f"Первая строка: {z[0, :]}")
print(f"Второй столбец: {z[:, 1]}")
print(f"Подматрица 2x2 в центре:\n{z[1:3, 1:3]}")

Первая строка: tensor([-0.5096, -2.6343, -0.5117, -0.1512])
Второй столбец: tensor([-2.6343,  0.7372, -0.9053,  1.1199])
Подматрица 2x2 в центре:
tensor([[ 0.7372,  0.6874],
        [-0.9053, -0.6006]])


In [ ]:
# Соединение тензоров
t_a = torch.ones(2, 2)
t_b = torch.zeros(2, 2)

# Конкатенация (склейка)
cat_v = torch.cat([t_a, t_b], dim=0) # По вертикали (4, 2)
cat_h = torch.cat([t_a, t_b], dim=1) # По горизонтали (2, 4)

# Стек (создание новой размерности)
stacked = torch.stack([t_a, t_b]) # Станет (2, 2, 2)

print(f"Shape после stack: {stacked.shape}")

Shape после stack: torch.Size([2, 2, 2])


Перцептрон

In [ ]:
# Подготовка данных
iris = load_iris()
X, y = iris.data, iris.target

# Берем только 2 класса для классического перцептрона (бинарная классификация)
X = X[y != 2]
y = y[y != 2]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Масштабирование — критически важно для нейросетей
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Конвертация в тензоры PyTorch
X_train_t = torch.FloatTensor(X_train)
print(y_train.shape)
y_train_t = torch.FloatTensor(y_train).reshape(-1, 1)
print(y_train_t.shape)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).reshape(-1, 1)

(80,)
torch.Size([80, 1])


In [ ]:
# Реализация Перцептрона
class Perceptron(nn.Module):
    def __init__(self, input_dim):
        super(Perceptron, self).__init__()
        self.fc = nn.Linear(input_dim, 1) # Один слой
        self.sigmoid = nn.Sigmoid()       # Функция активации для вероятности

    def forward(self, x):
        return self.sigmoid(self.fc(x))

In [ ]:
model = Perceptron(input_dim=4)
criterion = nn.BCELoss() # Binary Cross Entropy для бинарной задачи
optimizer = optim.SGD(model.parameters(), lr=0.1)

In [ ]:
# Цикл обучения
for epoch in range(100):
    optimizer.zero_grad()           # Обнуляем градиенты
    outputs = model(X_train_t)      # Forward pass
    loss = criterion(outputs, y_train_t)
    loss.backward()                 # Backward pass (расчет градиентов)
    optimizer.step()                # Обновление весов

    if (epoch+1) % 20 == 0:
        print(f'Epoch [{epoch+1}/100], Loss: {loss.item():.4f}')

Epoch [20/100], Loss: 0.1881
Epoch [40/100], Loss: 0.1047
Epoch [60/100], Loss: 0.0732
Epoch [80/100], Loss: 0.0566
Epoch [100/100], Loss: 0.0463


In [ ]:
from sklearn.metrics import accuracy_score
# Проверка
y_pred = model(X_test_t).round().detach().numpy()
accuracy_score1 = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy_score1}')

Accuracy: 1.0


Multi-Output сеть с разными активациями

In [ ]:
class MultiOutputNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(MultiOutputNet, self).__init__()

        # Общий скрытый слой
        self.hidden = nn.Linear(input_size, hidden_size)

        # Выход 1: Регрессия (например, цена объекта)
        self.reg_layer = nn.Linear(hidden_size, 1)

        # Выход 2: Классификация (например, тип объекта)
        self.clf_layer = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # Применяем ReLU в скрытом слое (отсекаем отрицательные значения)
        x = F.relu(self.hidden(x))

        # Выход регрессии. Часто без активации или Tanh/Sigmoid, если диапазон ограничен.
        # Оставим линейным для произвольных чисел.
        reg_output = self.reg_layer(x)

        # Выход классификации. Используем Softmax для распределения вероятностей.
        clf_output = F.softmax(self.clf_layer(x), dim=1)

        return reg_output, clf_output

# Лабораторная работа
Легенда:
Вернёмся к датасету California Housing Prices: https://www.kaggle.com/datasets/camnugent/california-housing-prices. Вам нужно создать нейросеть, которая по этим признакам одновременно:
Предсказывает рыночную стоимость (задача регрессии).
Определяет категорию жилья (эконом, стандарт, люкс — задача классификации, для этого необходимо создать вторую целевую переменную, определив пороги для разбиения на классы жилья и преобразовав данные).

Техническое задание:
Входной слой: 10 нейронов.
Скрытый слой: подобрать экспериментально число нейронов. Используйте функцию активации ReLU.

Выход 1 (Regression Head): 1 нейрон. Используйте Identity (без активации) или ReLU, если цена не может быть отрицательной.

Выход 2 (Classification Head): 3 нейрона. Используйте Softmax для получения распределения вероятностей.


In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

In [ ]:
# Загрузка и предобработка данных
digits = load_digits()
X = digits.data / 16.0  # Масштабируем пиксели в диапазон [0, 1]
y = digits.target

# Превращаем метки в One-Hot векторы (например, 3 -> [0,0,0,1,0,0,0,0,0,0])
y_onehot = np.eye(10)[y]

X_train, X_test, y_train, y_test = train_test_split(X, y_onehot, test_size=0.2, random_state=42)

In [ ]:
class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size, lr=0.1):
        # Инициализация весов (Xavier-like)
        self.w1 = np.random.randn(input_size, hidden_size) * np.sqrt(2/input_size)
        self.b1 = np.zeros((1, hidden_size))
        self.w2 = np.random.randn(hidden_size, output_size) * np.sqrt(2/hidden_size)
        self.b2 = np.zeros((1, output_size))
        self.lr = lr

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def softmax(self, x):
        exps = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exps / np.sum(exps, axis=1, keepdims=True)

    def forward(self, X):
        self.z1 = np.dot(X, self.w1) + self.b1
        self.a1 = self.sigmoid(self.z1)
        self.z2 = np.dot(self.a1, self.w2) + self.b2
        self.a2 = self.softmax(self.z2)
        return self.a2

    def train(self, X, y, epochs=1000):
        for epoch in range(epochs):
            # Forward pass
            output = self.forward(X)

            # Backpropagation
            m = y.shape[0]
            dz2 = output - y  # Градиент для Softmax + Cross-Entropy
            dw2 = np.dot(self.a1.T, dz2) / m
            db2 = np.sum(dz2, axis=0, keepdims=True) / m

            dz1 = np.dot(dz2, self.w2.T) * (self.a1 * (1 - self.a1)) # Градиент Sigmoid
            dw1 = np.dot(X.T, dz1) / m
            db1 = np.sum(dz1, axis=0, keepdims=True) / m

            # Обновление весов
            self.w1 -= self.lr * dw1
            self.b1 -= self.lr * db1
            self.w2 -= self.lr * dw2
            self.b2 -= self.lr * db2

    def predict(self, X):
        return np.argmax(self.forward(X), axis=1)

In [ ]:
# Обучение
nn = NeuralNetwork(input_size=64, hidden_size=32, output_size=10, lr=0.5)
nn.train(X_train, y_train, epochs=1500)

In [ ]:
# Оценка
test_preds = nn.predict(X_test)
y_test_labels = np.argmax(y_test, axis=1)
accuracy = np.mean(test_preds == y_test_labels)

print(f"Accuracy на тестовой выборке: {accuracy * 100:.2f}%")

Accuracy на тестовой выборке: 97.22%
